In [ ]:
!pip install gcloud
!gcloud auth application-default login

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.4/454.4 kB 12.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for gcloud: filename=gcloud-0.18.3-py3-none-any.whl size=602927 sha256=b9f26dfeb28b2d47b580070f7f3b09b2df451e8622c1d81ff37b577cc3e53c8a
  Stored in directory: /root/.cache/pip/wheels/2a/62/75/3d74209bfebb8805823ae74afa28653aa1ea76d8b5a9d741ff
Successfully built gcloud
Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fapplicationdefaultauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=46DIV60wTlF1qEEMpjbAYo9hNCQYWf&prompt=consent&token_usage=remote&access_type=offline&code_chal

In [ ]:
import pandas as pd
import numpy as np
import os
import pandas_gbq
from google.cloud import bigquery
import glob
import openpyxl

# Tratando os dados referentes a 2024

In [ ]:
df = pd.read_excel('/content/Base_MUNIC_2024_20251107.xlsx', sheet_name='Informática e comunicação')
df

,Cod Munic,Uf,Cod Uf,Desc Mun,Populacao,Faixa_populacao,Regiao,Mtic011,Mtic012,Mtic013,...,Mtic367,Mtic368,Mtic37,Mtic381,Mtic382,Mtic383,Mtic384,Mtic385,Mtic386,Mtic387
0,1100015,RO,11,Alta Floresta DOeste,22853,4 - 20001 até 50000,1 - Norte,Não,Não,Sim,...,Não,Não,Não,Sim,Não,Não,Sim,Não,Não,Não
1,1100023,RO,11,Ariquemes,108573,6 - 100001 até 500000,1 - Norte,Não,Não,Sim,...,-,-,Sim,Não,Não,Não,Não,Sim,Não,Não
2,1100031,RO,11,Cabixi,5690,2 - 5001 até 10000,1 - Norte,Não,Não,Sim,...,-,-,Não,-,-,-,-,-,-,Sim
3,1100049,RO,11,Cacoal,97637,5 - 50001 até 100000,1 - Norte,Não,Não,Sim,...,-,-,Sim,Sim,Não,Não,Não,Não,Não,Não
4,1100056,RO,11,Cerejeiras,16975,3 - 10001 até 20000,1 - Norte,Não,Não,Sim,...,Não,Não,Sim,Sim,Sim,Não,Não,Não,Não,Não
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5565,5222005,GO,52,Vianópolis,15476,3 - 10001 até 20000,5 - Centro-Oeste,Não,Não,Sim,...,-,-,Sim,Sim,Sim,Sim,Não,Não,Não,Não
5566,5222054,GO,52,Vicentinópolis,9077,2 - 5001 até 10000,5 - Centro-Oeste,Não,Não,Sim,...,-,Sim,Não,-,-,-,-,-,-,Sim
5567,5222203,GO,52,Vila Boa,4185,1 - Até 5000,5 - Centro-Oeste,Não,Não,Sim,...,Não,Não,Não,Sim,Sim,Não,Não,Não,Não,Não
5568,5222302,GO,52,Vila Propício,5982,2 - 5001 até 10000,5 - Centro-Oeste,Sim,Não,Sim,...,-,-,Não,Sim,Não,Sim,Sim,Não,Não,Não


In [ ]:
df = df[['Cod Munic','Desc Mun','Cod Uf', 'Mtic251','Mtic252', 'Mtic254','Mtic253']]
df

,Cod Munic,Desc Mun,Cod Uf,Mtic251,Mtic252,Mtic254,Mtic253
0,1100015,Alta Floresta DOeste,11,Não,Não,Não,Não
1,1100023,Ariquemes,11,Sim,Não,Não,Sim
2,1100031,Cabixi,11,Não,Não,Não,Não
3,1100049,Cacoal,11,Sim,Não,Sim,Sim
4,1100056,Cerejeiras,11,Sim,Não,Não,Não
...,...,...,...,...,...,...,...
5565,5222005,Vianópolis,52,Sim,Não,Não,Não
5566,5222054,Vicentinópolis,52,Não,Não,Não,Não
5567,5222203,Vila Boa,52,Sim,Sim,Sim,Sim
5568,5222302,Vila Propício,52,Não,Não,Sim,Não


In [ ]:
df =  df.rename(columns={
                        'Cod Munic': 'id_municipio',
                        'Desc Mun':'nome_municipio',
                        'Cod Uf': 'cod_uf',
                        'Mtic251': 'consulta_publica_cidadaos',
                        'Mtic252':'grupos_discussao_foruns',
                        'Mtic254':'votacao_orientar_decisoes',
                        'Mtic253':'enquete_interesse_prefeitura',
})
df

,id_municipio,nome_municipio,cod_uf,consulta_publica_cidadaos,grupos_discussao_foruns,votacao_orientar_decisoes,enquete_interesse_prefeitura
0,1100015,Alta Floresta DOeste,11,Não,Não,Não,Não
1,1100023,Ariquemes,11,Sim,Não,Não,Sim
2,1100031,Cabixi,11,Não,Não,Não,Não
3,1100049,Cacoal,11,Sim,Não,Sim,Sim
4,1100056,Cerejeiras,11,Sim,Não,Não,Não
...,...,...,...,...,...,...,...
5565,5222005,Vianópolis,52,Sim,Não,Não,Não
5566,5222054,Vicentinópolis,52,Não,Não,Não,Não
5567,5222203,Vila Boa,52,Sim,Sim,Sim,Sim
5568,5222302,Vila Propício,52,Não,Não,Sim,Não


In [ ]:
cod_uf = pd.read_csv('/content/MUNIC_ALL_2021 - Variáveis externas.csv', sep=',')[['UF','Cod UF']]


In [ ]:
x= cod_uf.pivot_table(columns=('UF','Cod UF'), aggfunc='size')


In [ ]:
cod_uf = pd.DataFrame(x).reset_index()[['UF','Cod UF']]

In [ ]:
df = df.merge(cod_uf, right_on='Cod UF',left_on='cod_uf')
df

,id_municipio,nome_municipio,cod_uf,consulta_publica_cidadaos,grupos_discussao_foruns,votacao_orientar_decisoes,enquete_interesse_prefeitura,UF,Cod UF
0,1100015,Alta Floresta DOeste,11,Não,Não,Não,Não,RO,11
1,1100023,Ariquemes,11,Sim,Não,Não,Sim,RO,11
2,1100031,Cabixi,11,Não,Não,Não,Não,RO,11
3,1100049,Cacoal,11,Sim,Não,Sim,Sim,RO,11
4,1100056,Cerejeiras,11,Sim,Não,Não,Não,RO,11
...,...,...,...,...,...,...,...,...,...
5565,5222005,Vianópolis,52,Sim,Não,Não,Não,GO,52
5566,5222054,Vicentinópolis,52,Não,Não,Não,Não,GO,52
5567,5222203,Vila Boa,52,Sim,Sim,Sim,Sim,GO,52
5568,5222302,Vila Propício,52,Não,Não,Sim,Não,GO,52


In [ ]:
df['ano']=2024

In [ ]:
df=df[['ano', 'UF','Cod UF', 'cod_uf', 'nome_municipio','id_municipio','consulta_publica_cidadaos', 'grupos_discussao_foruns',
       'votacao_orientar_decisoes', 'enquete_interesse_prefeitura']]
df

,ano,UF,Cod UF,cod_uf,nome_municipio,id_municipio,consulta_publica_cidadaos,grupos_discussao_foruns,votacao_orientar_decisoes,enquete_interesse_prefeitura
0,2024,RO,11,11,Alta Floresta DOeste,1100015,Não,Não,Não,Não
1,2024,RO,11,11,Ariquemes,1100023,Sim,Não,Não,Sim
2,2024,RO,11,11,Cabixi,1100031,Não,Não,Não,Não
3,2024,RO,11,11,Cacoal,1100049,Sim,Não,Sim,Sim
4,2024,RO,11,11,Cerejeiras,1100056,Sim,Não,Não,Não
...,...,...,...,...,...,...,...,...,...,...
5565,2024,GO,52,52,Vianópolis,5222005,Sim,Não,Não,Não
5566,2024,GO,52,52,Vicentinópolis,5222054,Não,Não,Não,Não
5567,2024,GO,52,52,Vila Boa,5222203,Sim,Sim,Sim,Sim
5568,2024,GO,52,52,Vila Propício,5222302,Não,Não,Sim,Não


In [ ]:
df = df.drop(['cod_uf'], axis=1)
df

,ano,UF,Cod UF,nome_municipio,id_municipio,consulta_publica_cidadaos,grupos_discussao_foruns,votacao_orientar_decisoes,enquete_interesse_prefeitura
0,2024,RO,11,Alta Floresta DOeste,1100015,Não,Não,Não,Não
1,2024,RO,11,Ariquemes,1100023,Sim,Não,Não,Sim
2,2024,RO,11,Cabixi,1100031,Não,Não,Não,Não
3,2024,RO,11,Cacoal,1100049,Sim,Não,Sim,Sim
4,2024,RO,11,Cerejeiras,1100056,Sim,Não,Não,Não
...,...,...,...,...,...,...,...,...,...
5565,2024,GO,52,Vianópolis,5222005,Sim,Não,Não,Não
5566,2024,GO,52,Vicentinópolis,5222054,Não,Não,Não,Não
5567,2024,GO,52,Vila Boa,5222203,Sim,Sim,Sim,Sim
5568,2024,GO,52,Vila Propício,5222302,Não,Não,Sim,Não


In [ ]:
df =  df.rename(columns={
                        'UF':'sigla_uf',
                        'Cod UF':'cod_uf'})
df

,ano,sigla_uf,cod_uf,nome_municipio,id_municipio,consulta_publica_cidadaos,grupos_discussao_foruns,votacao_orientar_decisoes,enquete_interesse_prefeitura
0,2024,RO,11,Alta Floresta DOeste,1100015,Não,Não,Não,Não
1,2024,RO,11,Ariquemes,1100023,Sim,Não,Não,Sim
2,2024,RO,11,Cabixi,1100031,Não,Não,Não,Não
3,2024,RO,11,Cacoal,1100049,Sim,Não,Sim,Sim
4,2024,RO,11,Cerejeiras,1100056,Sim,Não,Não,Não
...,...,...,...,...,...,...,...,...,...
5565,2024,GO,52,Vianópolis,5222005,Sim,Não,Não,Não
5566,2024,GO,52,Vicentinópolis,5222054,Não,Não,Não,Não
5567,2024,GO,52,Vila Boa,5222203,Sim,Sim,Sim,Sim
5568,2024,GO,52,Vila Propício,5222302,Não,Não,Sim,Não


In [ ]:
df.columns

Index(['ano', 'sigla_uf', 'cod_uf', 'nome_municipio', 'id_municipio',
       'consulta_publica_cidadaos', 'grupos_discussao_foruns',
       'votacao_orientar_decisoes', 'enquete_interesse_prefeitura'],
      dtype='object')

In [ ]:
df['ano'].unique()

array([2024])

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5570 entries, 0 to 5569
Data columns (total 9 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   ano                           5570 non-null   int64 
 1   sigla_uf                      5570 non-null   object
 2   cod_uf                        5570 non-null   int64 
 3   nome_municipio                5570 non-null   object
 4   id_municipio                  5570 non-null   int64 
 5   consulta_publica_cidadaos     5570 non-null   object
 6   grupos_discussao_foruns       5570 non-null   object
 7   votacao_orientar_decisoes     5570 non-null   object
 8   enquete_interesse_prefeitura  5570 non-null   object
dtypes: int64(3), object(6)
memory usage: 391.8+ KB


# Consumindo o ano de 2019 através do GBQ

In [ ]:


query = """SELECT * FROM `repositoriodedadosgpsp.participacao_transparencia.MUNIC_formas_participacao_cidadao_internet` WHERE ano = 2019"""
# Execute the query using pandas_gbq.read_gbq and load the result into a pandas DataFrame called 'df'.
# The 'project_id' specifies the Google Cloud Project to use.
df_2019 = pandas_gbq.read_gbq(query, project_id='repositoriodedadosgpsp')



/usr/local/lib/python3.12/dist-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Downloading: 100%|██████████|


In [ ]:
df_2019.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5570 entries, 0 to 5569
Data columns (total 9 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   ano                           5570 non-null   Int64 
 1   sigla_uf                      5570 non-null   object
 2   cod_uf                        5570 non-null   Int64 
 3   nome_municipio                5570 non-null   object
 4   id_municipio                  5570 non-null   Int64 
 5   consulta_publica_cidadaos     5570 non-null   object
 6   grupos_discussao_foruns       5570 non-null   object
 7   votacao_orientar_decisoes     5570 non-null   object
 8   enquete_interesse_prefeitura  5570 non-null   object
dtypes: Int64(3), object(6)
memory usage: 408.1+ KB


In [ ]:
df_2019['ano'].unique()

<IntegerArray>
[2019]
Length: 1, dtype: Int64

# Agregando os anos

In [ ]:
df_final = pd.concat([df, df_2019], ignore_index=True)

In [ ]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11140 entries, 0 to 11139
Data columns (total 9 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   ano                           11140 non-null  Int64 
 1   sigla_uf                      11140 non-null  object
 2   cod_uf                        11140 non-null  Int64 
 3   nome_municipio                11140 non-null  object
 4   id_municipio                  11140 non-null  Int64 
 5   consulta_publica_cidadaos     11140 non-null  object
 6   grupos_discussao_foruns       11140 non-null  object
 7   votacao_orientar_decisoes     11140 non-null  object
 8   enquete_interesse_prefeitura  11140 non-null  object
dtypes: Int64(3), object(6)
memory usage: 816.0+ KB


In [ ]:
df_final['ano'].unique()

<IntegerArray>
[2024, 2019]
Length: 2, dtype: Int64

Subindo para o GBQ

In [ ]:
# Import the bigquery library from google.cloud
from google.cloud import bigquery

# Initialize the BigQuery client, specifying the Google Cloud project ID.
# This client object is used to interact with the BigQuery API.
client = bigquery.Client(project='repositoriodedadosgpsp')
dataset_ref = client.dataset('participacao_transparencia')

/usr/local/lib/python3.12/dist-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [ ]:
schema=[bigquery.SchemaField('ano','INTEGER',description='Ano de referência da observação'),
    bigquery.SchemaField('sigla_uf','STRING',description='Sigla da Unidade da Federação'),
    bigquery.SchemaField('cod_uf','INTEGER',description='Código do IBGE da UF'),
    bigquery.SchemaField('nome_municipio','STRING',description='Nome do município da observação'),
    bigquery.SchemaField('id_municipio','INTEGER',description='Identificador do município pelo IBGE'),
    bigquery.SchemaField('consulta_publica_cidadaos','STRING',description='Consulta pública on line para que cidadãos possam enviar'),
    bigquery.SchemaField('grupos_discussao_foruns','STRING',description='Grupos de discussão como fóruns ou comunidades pela internet'),
    bigquery.SchemaField('votacao_orientar_decisoes','STRING',description='Votação on line para orientar a tomada de decisão'),
    bigquery.SchemaField('enquete_interesse_prefeitura','STRING',description='Enquete on line sobre assuntos de interesse do governo estadual ou municipal'),
]

In [ ]:
table_ref = dataset_ref.table('MUNIC_formas_participacao_cidadao_internet_v1')
job_config = bigquery.LoadJobConfig(schema=schema)
job = client.load_table_from_dataframe(df_final,table_ref, job_config=job_config)
job.result()

LoadJob<project=repositoriodedadosgpsp, location=US, id=698f01ad-b63f-4194-9091-65f5a0fbbe1d>